# LSTMs with PyTorch — Sine Wave Prediction

The vanilla RNN struggles with **long-range dependencies** because gradients shrink exponentially
as they travel back through time (the vanishing gradient problem).

**LSTMs** (Long Short-Term Memory networks) solve this with two states:
- **Hidden state** `h_t`: short-term memory (same role as in a vanilla RNN)
- **Cell state** `c_t`: long-term memory — a highway for gradients to flow unchanged

Three gates control information flow:
- **Forget gate**: what to erase from `c_{t-1}`
- **Input gate**: what new information to write to `c_t`
- **Output gate**: what to expose as `h_t`

In this exercise you'll predict sine wave values from a sliding window of past values.

## Step 1: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

print('PyTorch version:', torch.__version__)

## Step 2: Generate the Sine Wave Dataset

We generate 200 points of a sine wave. This is our time series.

In [ ]:
# TODO: use np.linspace to create t from 0 to 4*pi with 200 points
t = ...
# TODO: compute data = np.sin(t).astype(np.float32)
data = ...

print(f'Data shape: {data.shape}')
print(f'Min: {data.min():.3f}, Max: {data.max():.3f}')
print(f'First 5 values: {data[:5]}')

## Step 3: Create Sliding Window Sequences

With `seq_len=20`, each training sample is:
- **Input** `X[i]`: `data[i : i+seq_len]` — the last 20 values, shape `(seq_len, 1)`
- **Target** `y[i]`: `data[i+seq_len]` — the next value to predict, shape `(1,)`

The `1` in the last dimension indicates 1 feature (the sine value at that timestep).

In [ ]:
def create_sequences(data, seq_len=20):
    """
    Returns:
      X: shape (N, seq_len, 1) as float32 tensor
      y: shape (N, 1) as float32 tensor
    """
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        # TODO: append data[i:i+seq_len] reshaped to (seq_len, 1)
        xs.append(...)
        # TODO: append data[i+seq_len] reshaped to (1,)
        ys.append(...)
    # TODO: return torch tensors converted from numpy arrays
    return ..., ...

X, y = create_sequences(data, seq_len=20)
print('X shape:', X.shape)
print('y shape:', y.shape)

## Step 4: Train/Test Split

Split 80% for training, 20% for testing. Preserve temporal order — do NOT shuffle.

In [ ]:
# TODO: compute split_idx as int(0.8 * len(X))
split_idx = ...

# TODO: create X_train, X_test, y_train, y_test
X_train, X_test = ...
y_train, y_test = ...

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Step 5: Define the LSTM Model

Key difference from `nn.RNN`:
- `nn.LSTM` returns `(output, (h_n, c_n))` — a **tuple** for the hidden state
- `h_n` is the hidden state, `c_n` is the cell state
- We use `output[:, -1, :]` — the last timestep's output — for regression

With `num_layers=2`, the LSTM is stacked: output of layer 1 feeds into layer 2.

In [ ]:
class SineLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        # TODO: define self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.lstm = ...
        # TODO: define self.fc = nn.Linear(hidden_size, 1)
        self.fc = ...

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        # TODO: pass x through self.lstm
        # Unpack: out, (h_n, c_n) = self.lstm(x)
        out, (h_n, c_n) = ...

        # TODO: take last timestep: out[:, -1, :] shape (batch, hidden_size)
        last_out = ...

        # TODO: apply self.fc -> shape (batch, 1)
        return self.fc(last_out)

model = SineLSTM()
print(model)

## Step 6: Training Loop

In [ ]:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# TODO: train for 100 epochs
# Print train loss every 10 epochs
for epoch in range(100):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        # TODO: forward, loss, zero_grad, backward, step
        pass

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d} | Train Loss: {total_loss/len(train_loader):.6f}')

## Step 7: Evaluate on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    # TODO: get predictions on X_test
    test_preds = ...
    test_loss = criterion(test_preds, y_test)
    print(f'Test MSE: {test_loss.item():.6f}')

## Step 8: Visualize Predictions vs Actuals

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze().numpy()
    actuals = y_test.squeeze().numpy()

print('Predicted vs Actual (first 10):')
for i in range(10):
    print(f'  Pred: {preds[i]:.4f} | Actual: {actuals[i]:.4f}')

if HAS_MATPLOTLIB:
    plt.figure(figsize=(12, 4))
    plt.plot(actuals, label='Actual', linewidth=2)
    plt.plot(preds, label='Predicted', linewidth=2, linestyle='--')
    plt.legend()
    plt.title('LSTM Sine Wave Prediction')
    plt.xlabel('Time step')
    plt.ylabel('Value')
    plt.show()

## Step 9: Compare LSTM vs RNN

Rebuild the same architecture but use `nn.RNN` instead of `nn.LSTM`. Train and compare.

Note: `nn.RNN` returns `(output, h_n)` — no cell state tuple.

In [ ]:
class SineRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2):
        super().__init__()
        # TODO: define self.rnn = nn.RNN(...) mirroring SineLSTM
        self.rnn = ...
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        # TODO: out, h_n = self.rnn(x)
        out, h_n = ...
        return self.fc(out[:, -1, :])

rnn_model = SineRNN()
optimizer_rnn = optim.Adam(rnn_model.parameters(), lr=0.001)

# TODO: train rnn_model for 100 epochs (same loop as above)
for epoch in range(100):
    pass

# TODO: evaluate test MSE and compare
print('\nFinal comparison:')
print(f'  LSTM Test MSE: {test_loss.item():.6f}')
# print(f'  RNN  Test MSE: ...')

## Bonus: Multi-layer LSTM with Dropout

`nn.LSTM` supports `dropout` between layers (applied to all layers except the last).
Try adding dropout for regularization.

In [ ]:
class SineLSTMDeep(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: nn.LSTM with input_size=1, hidden_size=32, num_layers=3, dropout=0.2
        self.lstm = ...
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# TODO: train and compare test MSE with 2-layer version